
# ML Lab 4 — Improving and Comparing Machine Learning Models

**Name:** Abdul Qudoos 
**CMS:** 023-24-0233  
**Section:** G  
**GitHub Profile:**(https://github.com/abdulqudoos181005)
**Kaggle Profile:** https://www.kaggle.com/abdulqudoos7860 
 




## 1. Decision Tree Baseline

First, we load the cleaned dataset and create the same `X`, `y`, and train/test split used in Lab 3.


In [4]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, GridSearchCV

# Change this file name if your cleaned Lab 2 dataset has a different name
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Remove customer ID because it is only an identifier
df = df.drop(columns=["customerID"], errors="ignore")

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Convert Yes/No columns
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].replace({"Yes": 1, "No": 0})

# One-hot encode remaining text columns
df = pd.get_dummies(df, drop_first=True)

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X shape:", X.shape)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


KeyError: "['Churn'] not found in axis"

In [ ]:

# Use the max_depth that was selected as best in Lab 3
best_depth = 5

dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred, zero_division=0)
dt_recall = recall_score(y_test, dt_pred, zero_division=0)
dt_f1 = f1_score(y_test, dt_pred, zero_division=0)

print("Decision Tree Results")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1-score :", round(dt_f1, 4))


**Interpretation:** These four scores are our Decision Tree benchmark. We will use the same test data for a fair comparison with the Random Forest.


## 2. Random Forest

A Random Forest uses many Decision Trees instead of one tree. This usually makes the model more stable and can reduce overfitting.


In [ ]:

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest predictions created.")


**Task 4 — Why can Random Forest generalize better?** A Random Forest combines many trees, so one tree's mistakes have less effect. This makes the model more stable and can reduce overfitting.

## 3. Model Evaluation

In [ ]:

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, zero_division=0)
rf_recall = recall_score(y_test, rf_pred, zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, zero_division=0)

print("Random Forest Results")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1-score :", round(rf_f1, 4))

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred)
plt.title("Random Forest Confusion Matrix")
plt.show()


**Interpretation:** The confusion matrix shows how many customers were correctly and incorrectly classified as churn or not churn.

## 4. Decision Tree vs Random Forest

In [ ]:

comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [dt_accuracy, rf_accuracy],
    "Precision": [dt_precision, rf_precision],
    "Recall": [dt_recall, rf_recall],
    "F1-score": [dt_f1, rf_f1]
})

comparison.round(4)


**Task 7 — Interpretation:** Compare the two rows above. The model with the higher F1-score is better at balancing precision and recall. Because churn is an imbalanced class, F1-score and recall are more useful than accuracy alone.

## 5. Cross-Validation

In [ ]:

dt_cv = cross_val_score(dt, X, y, cv=5, scoring="f1")
rf_cv = cross_val_score(rf, X, y, cv=5, scoring="f1")

print("Decision Tree CV F1")
print("Mean:", round(dt_cv.mean(), 4))
print("Std :", round(dt_cv.std(), 4))

print("\nRandom Forest CV F1")
print("Mean:", round(rf_cv.mean(), 4))
print("Std :", round(rf_cv.std(), 4))


**Task 9 — Interpretation:** Cross-validation gives scores from several different splits, so it gives a more reliable estimate than one train/test split. If the same model has the higher mean F1, our confidence in that model increases.

## 6. Hyperparameter Tuning

In [ ]:

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1-score:", round(grid.best_score_, 4))

tuned_rf = grid.best_estimator_


**Interpretation:** GridSearchCV tried the small set of parameter combinations using 5-fold cross-validation. The reported parameters gave the best average F1-score.

In [ ]:

tuned_pred = tuned_rf.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_pred)
tuned_precision = precision_score(y_test, tuned_pred, zero_division=0)
tuned_recall = recall_score(y_test, tuned_pred, zero_division=0)
tuned_f1 = f1_score(y_test, tuned_pred, zero_division=0)

print("Tuned Random Forest Results")
print("Accuracy :", round(tuned_accuracy, 4))
print("Precision:", round(tuned_precision, 4))
print("Recall   :", round(tuned_recall, 4))
print("F1-score :", round(tuned_f1, 4))


**Interpretation:** These test-set scores show whether tuning actually improved the model on data it did not train on.

## 7. Final Model Comparison

In [ ]:

final_comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Tuned Random Forest"],
    "Accuracy": [dt_accuracy, rf_accuracy, tuned_accuracy],
    "Precision": [dt_precision, rf_precision, tuned_precision],
    "Recall": [dt_recall, rf_recall, tuned_recall],
    "F1-score": [dt_f1, rf_f1, tuned_f1]
})

final_comparison.round(4)



**Task 14 — Final model selection:** I would choose the model with the best test-set F1-score while also checking recall and precision. This is better than choosing only by training score because the test results show how well the model works on unseen data.


## 8. Feature Importance

In [ ]:

importances = pd.Series(
    tuned_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 5 important features:")
print(importances.head(5))

plt.figure(figsize=(8, 5))
importances.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.show()


**Interpretation:** The top 3–5 features are the features the final Random Forest found most useful for predicting churn. These should be compared with the important patterns found in Lab 2 EDA and the Lab 3 Decision Tree.

## 9. Final Model Selection

In [ ]:

# Select the model with the highest test F1-score
scores = {
    "Decision Tree": dt_f1,
    "Random Forest": rf_f1,
    "Tuned Random Forest": tuned_f1
}

final_model_name = max(scores, key=scores.get)

if final_model_name == "Decision Tree":
    final_model = dt
elif final_model_name == "Random Forest":
    final_model = rf
else:
    final_model = tuned_rf

print("Final model:", final_model_name)
print("Test F1-score:", round(scores[final_model_name], 4))


**Final choice:** The model selected above has the highest test F1-score among the three models. I chose it using unseen-test performance rather than training performance alone.

## 10. Kaggle Prediction / Submission

In [ ]:

# Generate predictions using the selected final model
final_pred = final_model.predict(X_test)

# Customer IDs are kept separately if they exist in the original dataset
original = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Use test indices to create a simple submission-style file.
# If your Lab 2/3 split has a customer ID column, replace this ID column
# with the matching customer IDs from your own test set.
submission = pd.DataFrame({
    "customerID": original.loc[X_test.index, "customerID"].values
        if "customerID" in original.columns else X_test.index,
    "Churn": final_pred
})

submission["Churn"] = submission["Churn"].map({1: "Yes", 0: "No"})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\nSaved as submission.csv")


**Interpretation:** The submission file contains a customer identifier and the predicted Churn value, following the format required by the lab.


## 11. Conclusion

The lab compared a Decision Tree with a default Random Forest and a tuned Random Forest. The Random Forest models were evaluated using accuracy, precision, recall, F1-score, and a confusion matrix. Cross-validation gave a more reliable view of model performance, while GridSearchCV helped find better Random Forest settings. The final model was selected using test-set evidence, especially F1-score because the churn classes are imbalanced. The model can still be improved by trying other algorithms, deeper tuning, or methods that directly handle class imbalance.

### Task 18 — What I would try next

If I had more time, I would try other models such as Logistic Regression, Gradient Boosting, or XGBoost. I would also test more hyperparameters and try techniques for handling class imbalance. I would then compare all models using the same evaluation metrics.
